# Risk Premium Assessment Engine Demo

This notebook demonstrates the capabilities of the Risk Premium Assessment Engine, which combines traditional credit scoring with Algorand-specific on-chain behavior analysis to determine appropriate risk premiums for loan pricing.

## Key Features
- Traditional credit scoring integration
- Algorand wallet transaction history analysis
- On-chain behavior scoring (DeFi participation, governance voting, staking)
- Cross-chain credit analysis
- Dynamic risk scoring based on market conditions
- Risk premium calculation with multiple factors

In [ ]:
# Import required libraries
import asyncio
import json
from datetime import datetime, timedelta
from decimal import Decimal
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import risk premium assessment components
from risk_premium_assessment.core import (
    RiskPremiumEngine,
    CreditProfile,
    AlgorandWalletProfile,
    CrossChainProfile,
    MarketConditions
)
from risk_premium_assessment.config import RiskPremiumConfig

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Initialize the Risk Premium Assessment Engine

In [ ]:
# Create configuration
config = RiskPremiumConfig()

# Customize some parameters for demo
config.risk_thresholds.min_wallet_age_days = 90
config.risk_thresholds.min_transaction_count = 10
config.enable_detailed_logging = True

# Initialize the engine
risk_engine = RiskPremiumEngine(config)

print("Risk Premium Assessment Engine initialized successfully!")
print(f"Configuration: {len(config.algorand_specific_params.defi_protocol_whitelist)} whitelisted DeFi protocols")

## 2. Create Sample Borrower Profiles

Let's create several different borrower profiles to demonstrate the engine's capabilities.

In [ ]:
# Excellent Credit + Active Algorand User
excellent_credit_profile = CreditProfile(
    credit_score=780,
    credit_age_months=72,  # 6 years
    credit_utilization=0.15,
    payment_history_score=0.98,
    credit_mix_score=0.85,
    new_credit_score=0.9,
    debt_to_income_ratio=0.25
)

excellent_wallet_profile = AlgorandWalletProfile(
    wallet_address="EXCELLENT_USER_WALLET_ADDRESS_1234567890ABCDEF123456",
    wallet_age_days=900,  # 2.5 years
    total_transactions=2500,
    total_volume_algo=Decimal('150000'),
    average_balance_algo=Decimal('8000'),
    governance_participation_count=6,
    governance_commitment_algo=Decimal('25000'),
    staking_periods=12,
    staking_rewards_algo=Decimal('800'),
    defi_protocols_used=['tinyman', 'algofi', 'folks_finance', 'pact'],
    smart_contracts_interacted=75,
    asa_tokens_created=3,
    consensus_participation_periods=8
)

print("Created excellent borrower profile:")
print(f"Credit Score: {excellent_credit_profile.credit_score}")
print(f"Wallet Age: {excellent_wallet_profile.wallet_age_days} days")
print(f"DeFi Protocols: {len(excellent_wallet_profile.defi_protocols_used)}")

In [ ]:
# Good Credit + Moderate Algorand User
good_credit_profile = CreditProfile(
    credit_score=720,
    credit_age_months=48,  # 4 years
    credit_utilization=0.35,
    payment_history_score=0.88,
    credit_mix_score=0.75,
    new_credit_score=0.8,
    debt_to_income_ratio=0.4
)

good_wallet_profile = AlgorandWalletProfile(
    wallet_address="GOOD_USER_WALLET_ADDRESS_567890ABCDEF1234567890ABCD",
    wallet_age_days=400,  # 13 months
    total_transactions=800,
    total_volume_algo=Decimal('45000'),
    average_balance_algo=Decimal('3500'),
    governance_participation_count=2,
    governance_commitment_algo=Decimal('8000'),
    staking_periods=4,
    staking_rewards_algo=Decimal('200'),
    defi_protocols_used=['tinyman', 'algofi'],
    smart_contracts_interacted=25,
    asa_tokens_created=1
)

print("Created good borrower profile:")
print(f"Credit Score: {good_credit_profile.credit_score}")
print(f"Wallet Age: {good_wallet_profile.wallet_age_days} days")

In [ ]:
# Fair Credit + New Algorand User
fair_credit_profile = CreditProfile(
    credit_score=620,
    credit_age_months=24,  # 2 years
    credit_utilization=0.65,
    payment_history_score=0.75,
    credit_mix_score=0.6,
    new_credit_score=0.5,
    debt_to_income_ratio=0.55
)

fair_wallet_profile = AlgorandWalletProfile(
    wallet_address="FAIR_USER_WALLET_ADDRESS_890ABCDEF1234567890ABCDEF12",
    wallet_age_days=120,  # 4 months
    total_transactions=150,
    total_volume_algo=Decimal('8000'),
    average_balance_algo=Decimal('1200'),
    governance_participation_count=0,
    governance_commitment_algo=Decimal('0'),
    staking_periods=0,
    staking_rewards_algo=Decimal('0'),
    defi_protocols_used=['tinyman'],
    smart_contracts_interacted=8,
    asa_tokens_created=0
)

print("Created fair borrower profile:")
print(f"Credit Score: {fair_credit_profile.credit_score}")
print(f"Wallet Age: {fair_wallet_profile.wallet_age_days} days")

## 3. Define Market Conditions

Let's create different market scenarios to test the engine's response to varying conditions.

In [ ]:
# Normal Market Conditions
normal_market = MarketConditions(
    volatility_index=0.25,
    liquidity_ratio=0.85,
    correlation_factor=0.3,
    market_stress_indicator=0.15,
    algo_price_volatility=0.35
)

# Stressed Market Conditions
stressed_market = MarketConditions(
    volatility_index=0.65,
    liquidity_ratio=0.4,
    correlation_factor=0.8,
    market_stress_indicator=0.75,
    algo_price_volatility=0.8
)

# Bull Market Conditions
bull_market = MarketConditions(
    volatility_index=0.15,
    liquidity_ratio=0.95,
    correlation_factor=0.2,
    market_stress_indicator=0.05,
    algo_price_volatility=0.2
)

print("Created market condition scenarios:")
print(f"Normal: Volatility {normal_market.volatility_index}, Liquidity {normal_market.liquidity_ratio}")
print(f"Stressed: Volatility {stressed_market.volatility_index}, Liquidity {stressed_market.liquidity_ratio}")
print(f"Bull: Volatility {bull_market.volatility_index}, Liquidity {bull_market.liquidity_ratio}")

## 4. Run Risk Assessments

Now let's run comprehensive risk assessments for each borrower profile under different market conditions.

In [ ]:
async def run_assessments():
    """Run all risk assessments."""
    assessments = []
    
    # Define borrower profiles
    profiles = [
        ("Excellent", excellent_credit_profile, excellent_wallet_profile),
        ("Good", good_credit_profile, good_wallet_profile),
        ("Fair", fair_credit_profile, fair_wallet_profile)
    ]
    
    # Define market conditions
    markets = [
        ("Normal", normal_market),
        ("Bull", bull_market),
        ("Stressed", stressed_market)
    ]
    
    for profile_name, credit_prof, wallet_prof in profiles:
        for market_name, market_cond in markets:
            borrower_id = f"{profile_name.lower()}_{market_name.lower()}_001"
            
            assessment = await risk_engine.assess_borrower_risk(
                borrower_id=borrower_id,
                credit_profile=credit_prof,
                wallet_profile=wallet_prof,
                market_conditions=market_cond
            )
            
            assessments.append({
                'profile': profile_name,
                'market': market_name,
                'borrower_id': borrower_id,
                'assessment': assessment
            })
    
    return assessments

# Run assessments
assessment_results = await run_assessments()
print(f"Completed {len(assessment_results)} risk assessments")

## 5. Analyze Results

Let's create visualizations and analysis of the assessment results.

In [ ]:
# Create DataFrame for analysis
results_data = []

for result in assessment_results:
    assessment = result['assessment']
    results_data.append({
        'Profile': result['profile'],
        'Market': result['market'],
        'Credit_Score_Component': assessment.credit_score_component,
        'OnChain_Behavior_Component': assessment.onchain_behavior_component,
        'Cross_Chain_Component': assessment.cross_chain_component,
        'Market_Adjustment': assessment.market_adjustment_component,
        'Final_Risk_Score': assessment.final_risk_score,
        'Risk_Category': assessment.risk_category,
        'Risk_Premium_Rate': float(assessment.risk_premium_rate),
        'Recommended_Interest_Rate': float(assessment.recommended_interest_rate),
        'Confidence_Score': assessment.confidence_score
    })

df = pd.DataFrame(results_data)
print("Results DataFrame created:")
print(df.head())

In [ ]:
# Create risk premium heatmap
plt.figure(figsize=(12, 8))

# Pivot data for heatmap
heatmap_data = df.pivot(index='Profile', columns='Market', values='Risk_Premium_Rate')

# Create heatmap
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', 
            cbar_kws={'label': 'Risk Premium Rate'})
plt.title('Risk Premium Rates by Borrower Profile and Market Conditions')
plt.ylabel('Borrower Profile')
plt.xlabel('Market Conditions')
plt.tight_layout()
plt.show()

In [ ]:
# Component score breakdown
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Credit Score Component
sns.barplot(data=df, x='Profile', y='Credit_Score_Component', hue='Market', ax=axes[0,0])
axes[0,0].set_title('Credit Score Component by Profile and Market')
axes[0,0].set_ylim(0, 1)

# On-Chain Behavior Component
sns.barplot(data=df, x='Profile', y='OnChain_Behavior_Component', hue='Market', ax=axes[0,1])
axes[0,1].set_title('On-Chain Behavior Component by Profile and Market')
axes[0,1].set_ylim(0, 1)

# Market Adjustment
sns.barplot(data=df, x='Profile', y='Market_Adjustment', hue='Market', ax=axes[1,0])
axes[1,0].set_title('Market Adjustment Factor by Profile and Market')

# Final Risk Score
sns.barplot(data=df, x='Profile', y='Final_Risk_Score', hue='Market', ax=axes[1,1])
axes[1,1].set_title('Final Risk Score by Profile and Market')
axes[1,1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Risk category distribution
plt.figure(figsize=(12, 6))

# Count risk categories
risk_category_counts = df['Risk_Category'].value_counts()

# Create pie chart
plt.subplot(1, 2, 1)
plt.pie(risk_category_counts.values, labels=risk_category_counts.index, autopct='%1.1f%%')
plt.title('Distribution of Risk Categories')

# Interest rate by profile
plt.subplot(1, 2, 2)
sns.boxplot(data=df, x='Profile', y='Recommended_Interest_Rate')
plt.title('Recommended Interest Rates by Profile')
plt.ylabel('Interest Rate')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 6. Detailed Assessment Analysis

Let's examine one assessment in detail to understand the scoring breakdown.

In [ ]:
# Select excellent profile under normal market conditions for detailed analysis
detailed_assessment = None
for result in assessment_results:
    if result['profile'] == 'Excellent' and result['market'] == 'Normal':
        detailed_assessment = result['assessment']
        break

if detailed_assessment:
    print("Detailed Assessment Report:")
    print("=" * 50)
    report = risk_engine.generate_assessment_report(detailed_assessment)
    print(report)
else:
    print("Assessment not found")

In [ ]:
# Visualize component breakdown for excellent user
if detailed_assessment and 'credit_breakdown' in detailed_assessment.assessment_details:
    credit_breakdown = detailed_assessment.assessment_details['credit_breakdown']
    onchain_breakdown = detailed_assessment.assessment_details['onchain_breakdown']
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Credit score breakdown
    credit_labels = list(credit_breakdown.keys())
    credit_values = list(credit_breakdown.values())
    
    axes[0].bar(credit_labels, credit_values)
    axes[0].set_title('Credit Score Component Breakdown')
    axes[0].set_ylabel('Score')
    axes[0].set_ylim(0, 1)
    axes[0].tick_params(axis='x', rotation=45)
    
    # On-chain behavior breakdown
    onchain_labels = list(onchain_breakdown.keys())
    onchain_values = list(onchain_breakdown.values())
    
    axes[1].bar(onchain_labels, onchain_values, color='orange')
    axes[1].set_title('On-Chain Behavior Component Breakdown')
    axes[1].set_ylabel('Score')
    axes[1].set_ylim(0, 1)
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 7. Market Sensitivity Analysis

Let's analyze how sensitive the risk assessments are to different market conditions.

In [ ]:
# Market sensitivity for excellent profile
excellent_results = df[df['Profile'] == 'Excellent'].copy()

print("Market Sensitivity Analysis - Excellent Profile:")
print("-" * 50)

for _, row in excellent_results.iterrows():
    print(f"{row['Market']} Market:")
    print(f"  Risk Premium: {row['Risk_Premium_Rate']:.3%}")
    print(f"  Interest Rate: {row['Recommended_Interest_Rate']:.3%}")
    print(f"  Market Adjustment: {row['Market_Adjustment']:.2f}x")
    print(f"  Risk Category: {row['Risk_Category']}")
    print()

# Calculate sensitivity metrics
normal_premium = excellent_results[excellent_results['Market'] == 'Normal']['Risk_Premium_Rate'].iloc[0]
stressed_premium = excellent_results[excellent_results['Market'] == 'Stressed']['Risk_Premium_Rate'].iloc[0]
bull_premium = excellent_results[excellent_results['Market'] == 'Bull']['Risk_Premium_Rate'].iloc[0]

stress_impact = (stressed_premium - normal_premium) / normal_premium
bull_impact = (bull_premium - normal_premium) / normal_premium

print(f"Stress Market Impact: {stress_impact:.1%} increase in premium")
print(f"Bull Market Impact: {bull_impact:.1%} change in premium")

## 8. On-Chain Behavior Impact Analysis

Let's analyze how different on-chain behaviors impact risk assessment.

In [ ]:
async def analyze_onchain_impact():
    """Analyze impact of different on-chain behaviors."""
    
    # Base profile with no on-chain activity
    base_wallet = AlgorandWalletProfile(
        wallet_address="BASE_WALLET_1234567890ABCDEF1234567890ABCDEF12345",
        wallet_age_days=200,
        total_transactions=50,
        total_volume_algo=Decimal('5000')
    )
    
    # Governance participant
    governance_wallet = AlgorandWalletProfile(
        wallet_address="GOVERNANCE_WALLET_1234567890ABCDEF1234567890ABCD",
        wallet_age_days=200,
        total_transactions=50,
        total_volume_algo=Decimal('5000'),
        governance_participation_count=3,
        governance_commitment_algo=Decimal('10000')
    )
    
    # DeFi user
    defi_wallet = AlgorandWalletProfile(
        wallet_address="DEFI_WALLET_1234567890ABCDEF1234567890ABCDEF123",
        wallet_age_days=200,
        total_transactions=50,
        total_volume_algo=Decimal('5000'),
        defi_protocols_used=['tinyman', 'algofi', 'folks_finance']
    )
    
    # Staking user
    staking_wallet = AlgorandWalletProfile(
        wallet_address="STAKING_WALLET_1234567890ABCDEF1234567890ABCDEF12",
        wallet_age_days=200,
        total_transactions=50,
        total_volume_algo=Decimal('5000'),
        staking_periods=6,
        staking_rewards_algo=Decimal('300')
    )
    
    wallets = [
        ("Base", base_wallet),
        ("Governance", governance_wallet),
        ("DeFi", defi_wallet),
        ("Staking", staking_wallet)
    ]
    
    # Standard credit profile for all
    standard_credit = CreditProfile(
        credit_score=700,
        credit_age_months=36,
        credit_utilization=0.3,
        payment_history_score=0.85
    )
    
    results = []
    for wallet_type, wallet_profile in wallets:
        assessment = await risk_engine.assess_borrower_risk(
            borrower_id=f"onchain_analysis_{wallet_type.lower()}",
            credit_profile=standard_credit,
            wallet_profile=wallet_profile,
            market_conditions=normal_market
        )
        
        results.append({
            'type': wallet_type,
            'onchain_score': assessment.onchain_behavior_component,
            'final_score': assessment.final_risk_score,
            'premium': float(assessment.risk_premium_rate),
            'category': assessment.risk_category
        })
    
    return results

onchain_results = await analyze_onchain_impact()

# Display results
print("On-Chain Behavior Impact Analysis:")
print("-" * 50)
for result in onchain_results:
    print(f"{result['type']} User:")
    print(f"  On-Chain Score: {result['onchain_score']:.3f}")
    print(f"  Final Risk Score: {result['final_score']:.3f}")
    print(f"  Risk Premium: {result['premium']:.3%}")
    print(f"  Risk Category: {result['category']}")
    print()

In [ ]:
# Visualize on-chain behavior impact
onchain_df = pd.DataFrame(onchain_results)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# On-chain scores
axes[0].bar(onchain_df['type'], onchain_df['onchain_score'], color='skyblue')
axes[0].set_title('On-Chain Behavior Scores')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)

# Final risk scores
axes[1].bar(onchain_df['type'], onchain_df['final_score'], color='lightgreen')
axes[1].set_title('Final Risk Scores')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1)

# Risk premiums
axes[2].bar(onchain_df['type'], onchain_df['premium'], color='salmon')
axes[2].set_title('Risk Premiums')
axes[2].set_ylabel('Premium Rate')

plt.tight_layout()
plt.show()

# Calculate improvements
base_premium = onchain_df[onchain_df['type'] == 'Base']['premium'].iloc[0]
for _, row in onchain_df.iterrows():
    if row['type'] != 'Base':
        improvement = (base_premium - row['premium']) / base_premium
        print(f"{row['type']} activity provides {improvement:.1%} premium reduction")

## 9. Configuration Impact Analysis

Let's see how different configuration parameters affect the risk assessment.

In [ ]:
async def analyze_config_impact():
    """Analyze impact of different configuration parameters."""
    
    # Conservative configuration
    conservative_config = RiskPremiumConfig()
    conservative_config.onchain_behavior_weights.defi_participation = 0.15  # Reduced
    conservative_config.onchain_behavior_weights.governance_voting = 0.10   # Reduced
    conservative_config.risk_premium_rates.base_rate = Decimal('0.03')     # Higher base
    
    # Aggressive configuration
    aggressive_config = RiskPremiumConfig()
    aggressive_config.onchain_behavior_weights.defi_participation = 0.30   # Increased
    aggressive_config.onchain_behavior_weights.governance_voting = 0.25    # Increased
    aggressive_config.risk_premium_rates.base_rate = Decimal('0.015')     # Lower base
    
    # Create engines with different configs
    conservative_engine = RiskPremiumEngine(conservative_config)
    aggressive_engine = RiskPremiumEngine(aggressive_config)
    
    # Test with excellent profile
    test_results = []
    engines = [
        ("Default", risk_engine),
        ("Conservative", conservative_engine),
        ("Aggressive", aggressive_engine)
    ]
    
    for config_name, engine in engines:
        assessment = await engine.assess_borrower_risk(
            borrower_id=f"config_test_{config_name.lower()}",
            credit_profile=excellent_credit_profile,
            wallet_profile=excellent_wallet_profile,
            market_conditions=normal_market
        )
        
        test_results.append({
            'config': config_name,
            'onchain_score': assessment.onchain_behavior_component,
            'final_score': assessment.final_risk_score,
            'premium': float(assessment.risk_premium_rate),
            'interest_rate': float(assessment.recommended_interest_rate)
        })
    
    return test_results

config_results = await analyze_config_impact()

print("Configuration Impact Analysis:")
print("-" * 50)
for result in config_results:
    print(f"{result['config']} Configuration:")
    print(f"  On-Chain Score: {result['onchain_score']:.3f}")
    print(f"  Final Score: {result['final_score']:.3f}")
    print(f"  Risk Premium: {result['premium']:.3%}")
    print(f"  Interest Rate: {result['interest_rate']:.3%}")
    print()

## 10. Summary and Insights

Let's summarize the key insights from our risk premium assessment analysis.

In [ ]:
print("RISK PREMIUM ASSESSMENT ENGINE - KEY INSIGHTS")
print("=" * 60)
print()

# Overall statistics
print("1. OVERALL STATISTICS:")
print(f"   • Risk premium range: {df['Risk_Premium_Rate'].min():.3%} - {df['Risk_Premium_Rate'].max():.3%}")
print(f"   • Average confidence score: {df['Confidence_Score'].mean():.1%}")
print(f"   • Most common risk category: {df['Risk_Category'].mode().iloc[0]}")
print()

# Profile analysis
print("2. BORROWER PROFILE IMPACT:")
profile_stats = df.groupby('Profile')[['Risk_Premium_Rate', 'Final_Risk_Score']].mean()
for profile in profile_stats.index:
    print(f"   • {profile}: {profile_stats.loc[profile, 'Risk_Premium_Rate']:.3%} avg premium, "
          f"{profile_stats.loc[profile, 'Final_Risk_Score']:.3f} avg risk score")
print()

# Market impact
print("3. MARKET CONDITION IMPACT:")
market_stats = df.groupby('Market')[['Market_Adjustment', 'Risk_Premium_Rate']].mean()
for market in market_stats.index:
    print(f"   • {market}: {market_stats.loc[market, 'Market_Adjustment']:.2f}x adjustment, "
          f"{market_stats.loc[market, 'Risk_Premium_Rate']:.3%} avg premium")
print()

# On-chain behavior insights
print("4. ON-CHAIN BEHAVIOR INSIGHTS:")
onchain_impact = df.groupby('Profile')['OnChain_Behavior_Component'].mean()
for profile in onchain_impact.index:
    print(f"   • {profile} profile on-chain score: {onchain_impact[profile]:.3f}")
print()

# Key recommendations
print("5. KEY RECOMMENDATIONS:")
print("   • On-chain activity significantly impacts risk assessment")
print("   • Governance participation provides meaningful risk reduction")
print("   • Market conditions can double risk premiums in stress scenarios")
print("   • DeFi protocol participation demonstrates financial sophistication")
print("   • Staking behavior indicates long-term commitment to ecosystem")
print()

print("Analysis complete! The Risk Premium Assessment Engine successfully")
print("combines traditional credit metrics with Algorand-specific on-chain")
print("behavior to provide sophisticated risk-based pricing.")